# Index-Enhanced Fund Analysis (Wind)

Benchmarks the 中证500/800/1000/2000 and 国证2000 active-quant / index-enhanced funds (AUM > 2亿, inception > 1 year) against their respective benchmark indices.

## Setup: connect to WindPy

In [ ]:
import os ;\
import site ;\
import sys ;\
os.system('mkdir -p ' + site.getusersitepackages()) ;\
os.system('ln -sf "/Applications/Wind API.app/Contents/python/WindPy.py"' + ' ' + site.getusersitepackages()) ;\
os.system('ln -sf "/Applications/Wind API.app/Contents/python/WindPy.py"' + ' ' + site.getsitepackages()[0]) ;\
os.system('rm -rf ~/.Wind') ;\
os.system('ln -sf ~/Library/Containers/com.wind.mac.api/Data/.Wind ~/.Wind') ;\
print("Current Python Version: " + sys.version) ;\
print("Current Python Env: " + sys.executable) ;\
print("WindPy installed at: " + site.getusersitepackages()) ;\
print("WindPy installed at: " + site.getsitepackages()[0])

In [ ]:
from WindPy import *

ret = w.start()
print(ret)

ret = w.isconnected()
print(ret)

#test WSS function
ret = w.wss("000001.SZ", "sec_name","")
print(ret)

## Imports

In [ ]:
import os
import pandas as pd
import numpy as np


## Fund universe: codes by benchmark group

In [ ]:
# 中证500/800/1000/2000 国证1000/2000为基准的主动量化和指数增强基金（aum大于2亿，成立时间大于一年）

# fund code for 中证500 
fund_codes_CSI500 = [
    "006729.OF", "006730.OF", "014305.OF", "014306.OF", "004945.OF", "561550.OF", "001556.OF", "001557.OF", "159610.OF",
    "006682.OF", "016935.OF", "012080.OF", "012081.OF", "002316.OF", "006048.OF", "007413.OF", "003016.OF", "014344.OF", "014155.OF", "003578.OF",
    "003986.OF", "014156.OF", "006593.OF", "006594.OF", "502000.OF", "009300.OF", "015508.OF", "005994.OF", "002906.OF", "007089.OF", "002907.OF", "005062.OF",
    "000478.OF", "161017.OF", "004192.OF", "013332.OF", "004193.OF", "001050.OF", "016854.OF", "021186.OF"
]

# fund code for 中证800
fund_codes_CSI800 = ["016276.OF", "022513.OF", "010673.OF"]

# fund code for 中证1000
fund_codes_CSI1000 = [
    "018013.OF", "018014.OF", "015466.OF", "017094.OF", "017095.OF", "017953.OF", "017954.OF", "019240.OF", "019241.OF", "005313.OF",
    "017644.OF", "005314.OF", "017645.OF", "017846.OF", "006165.OF", "015867.OF", "017847.OF", "006166.OF", "015868.OF", "019555.OF", "014201.OF",
    "014202.OF", "018157.OF", "018158.OF", "159680.OF", "161039.OF", "013331.OF", "017919.OF", "017920.OF", "016936.OF", "016937.OF", "016785.OF", "016942.OF", "016943.OF",
    "004194.OF", "004195.OF", "015784.OF"]
    
# fund code for 中证2000
fund_codes_CSI2000 = ["019918.OF", "019919.OF", "159552.OF"]

# no fund for 国证1000 

# fund code for 国证2000
fund_codes_CNI2000 = ["019318.OF", "019319.OF", "018653.OF"]

In [ ]:
# 主动量化基金 (active quant, not index-enhanced) benchmarked against 中证500
# NOTE: 012080.OF and 005994.OF also appear in fund_codes_CSI500 above (both are
# named as index-enhanced funds) - flagging in case that's a categorization mix-up.
fund_codes_ACTIVE_QUANT_CSI500 = [
    "007808.OF",
    "010779.OF",
    "006195.OF",
    "014805.OF",
    "001421.OF",
    "004641.OF",
    "004695.OF",
    "014135.OF",
    "001917.OF",
    "001244.OF",
    "013166.OF",
    "001990.OF",
    "006648.OF",
    "007831.OF",
    "011934.OF",
    "460009.OF",
    "021692.OF",
    "166107.OF",
    "012080.OF",
    "000006.OF",
    "001637.OF",
    "022909.OF",
    "005994.OF",
    "000978.OF",
    "014691.OF",
    "017715.OF",
    "161038.OF",
    "016267.OF",
    "004250.OF",
    "233009.OF",
    "014187.OF",
    "003717.OF",
    "006336.OF",
    "163110.OF",
    "011824.OF",
    "005120.OF",
    "011410.OF",
    "019561.OF",
    "010120.OF",
    "005189.OF",
    "005126.OF",
    "012977.OF",
    "020117.OF",
    "009043.OF",
    "002270.OF",
    "003241.OF",
    "018705.OF",
    "021404.OF",
    "005865.OF",
    "009874.OF",
    "011589.OF",
    "025984.OF",
    "025982.OF",
    "025386.OF",
    "025011.OF",
    "025013.OF",
    "025466.OF",
    "025447.OF"
]


## Data fetching functions

In [ ]:
def get_index_daily_returns(index_code, start_date, end_date, save_path=None, force_refresh=False):
    """
    Wind's w.wsd(usedf=True) normally returns a frame indexed by date, one
    column per field. For a single-day request (start_date == end_date) it
    sometimes collapses that and returns the frame TRANSPOSED instead -
    indexed by the security code, with the date as the column - since
    there's only one row either way. That orientation silently breaks
    anything downstream that treats the index as dates (e.g. the chunked
    fetch concatenates many of these and sorts by date), so it's corrected
    back here right after fetching, and also on cache load in case a
    chunk file was already saved in the bad orientation from before this
    fix existed.
    """
    if save_path and not force_refresh and os.path.exists(save_path):
        print(f"Loading cached index data from {save_path} (skip Wind fetch to save API quota)")
        cached = pd.read_csv(save_path, index_col=0)
        if index_code in cached.index:
            cached = cached.T
        return cached

    error_code, data = w.wsd(index_code, "pct_chg", start_date, end_date, "", usedf=True)

    if error_code != 0:
        print(error_code)
        print(data)
        return None

    if index_code in data.index:
        data = data.T

    if save_path:
        data.to_csv(save_path, index=True)

    return data


In [ ]:
def get_fund_nav_data(fund_codes, start_date, end_date, save_path=None, force_refresh=False):
      if save_path and not force_refresh and os.path.exists(save_path):
            print(f"Loading cached NAV data from {save_path} (skip Wind fetch to save API quota)")
            return pd.read_csv(save_path, index_col=0)

      w.start()

      error_code, nav_data = w.wsd(
            fund_codes,
            "NAV_acc",  # cumulative NAV: avoids artificial drops on fund distribution dates
            start_date,
            end_date,
            "",
            usedf=True,
      )

      name_error_code, name_data = w.wss(
            fund_codes,
            "sec_name",
            usedf=True,
      )

      if error_code != 0:
            print(error_code)
            print(nav_data)
            return None

      if name_error_code != 0:
            print(name_error_code)
            print(name_data)
            return None

      chart = nav_data.T
      chart.index.name = "fund_code"
      chart.columns = [pd.Timestamp(col).strftime("%Y-%m-%d") for col in chart.columns]

      fund_names = name_data.iloc[:, 0].reindex(chart.index)
      chart = pd.concat([fund_names.rename("fund_name"), chart], axis=1)

      if save_path:
            chart.to_csv(save_path, index=True)

      return chart


def calculate_fund_daily_returns(fund_nav_data, save_path=None):
      if isinstance(fund_nav_data, str):
            fund_nav_data = pd.read_csv(fund_nav_data, index_col=0)
      else:
            fund_nav_data = fund_nav_data.copy()

      fund_names = fund_nav_data["fund_name"]
      nav_data = fund_nav_data.drop(columns=["fund_name"]).copy()
      # Drop only the first date column, which pct_change always makes NaN for
      # every fund (no prior day to diff against). Do NOT use dropna(axis=1) here -
      # that would drop any date where even one fund lacks data (e.g. a fund that
      # launched partway through the window), truncating the whole group's date
      # range to only the period after the last fund to launch. Individual funds
      # simply carry NaN before their own inception, which downstream mean/std
      # calls already handle correctly per-fund (skipna is the pandas default).
      calculated_returns = nav_data.pct_change(axis=1).iloc[:, 1:] * 100
      calculated_returns = pd.concat(
            [fund_names.reindex(calculated_returns.index).rename("fund_name"), calculated_returns],
            axis=1,
      )

      if save_path:
            calculated_returns.to_csv(save_path, index=True)

      return calculated_returns


## Generalize across all benchmark groups + risk/ranking metrics

In [ ]:
# Map each benchmark group to its fund codes and index code.
#
# NOTE: the original `index_codes` list had 000852.SH (CSI1000) listed twice -
# once for CSI1000 and once for CSI2000 - which looks like a copy/paste bug.

FUND_GROUPS = {
    "CSI500":  {"fund_codes": fund_codes_CSI500,  "index_code": "000905.SH"},
    "CSI800":  {"fund_codes": fund_codes_CSI800,  "index_code": "000906.SH"},
    "CSI1000": {"fund_codes": fund_codes_CSI1000, "index_code": "000852.SH"},
    "CSI2000": {"fund_codes": fund_codes_CSI2000, "index_code": "932000.CSI"},
    "CNI2000": {"fund_codes": fund_codes_CNI2000, "index_code": "399303.SZ"},
}


In [ ]:
def calculate_excess_returns(fund_daily_returns, benchmark_daily_returns, save_path=None):
    """Daily excess return (%) = fund daily return - benchmark daily return, per fund."""
    fund_names = fund_daily_returns["fund_name"]
    returns = fund_daily_returns.drop(columns=["fund_name"])

    benchmark = benchmark_daily_returns.copy()
    benchmark.index = [pd.Timestamp(idx).strftime("%Y-%m-%d") for idx in benchmark.index]
    benchmark_row = benchmark.iloc[:, 0].reindex(returns.columns)

    excess = returns.sub(benchmark_row, axis=1)
    excess = pd.concat([fund_names.reindex(excess.index), excess], axis=1)

    if save_path:
        excess.to_csv(save_path, index=True)

    return excess


def calculate_max_drawdown(fund_nav_data, save_path=None):
    """Max drawdown (%) per fund, computed from NAV levels (not from returns)."""
    fund_names = fund_nav_data["fund_name"]
    nav = fund_nav_data.drop(columns=["fund_name"]).astype(float)

    running_max = nav.cummax(axis=1)
    drawdown = (nav - running_max) / running_max * 100
    max_dd = drawdown.min(axis=1)

    result = pd.DataFrame({"fund_name": fund_names, "max_drawdown_pct": max_dd})

    if save_path:
        result.to_csv(save_path, index=True)

    return result


def calculate_excess_max_drawdown(fund_daily_returns, benchmark_daily_returns, save_path=None):
    """
    Max drawdown (%) of CUMULATIVE EXCESS RETURN (fund vs. its own benchmark),
    not of NAV. Raw NAV drawdown mostly reflects beta/market drawdown - if the
    market falls 14%, a ~beta-1 enhanced fund falls ~14% too, which says
    nothing about the enhancement. This builds a synthetic cumulative index
    from the fund's daily excess-return series and measures its own
    peak-to-trough decline: how much the fund underperformed its benchmark at
    its worst point, which is the risk metric that actually matters for an
    "enhanced" product.
    """
    fund_names = fund_daily_returns["fund_name"]
    excess_pct = calculate_excess_returns(fund_daily_returns, benchmark_daily_returns)
    excess = excess_pct.drop(columns=["fund_name"]) / 100

    cum_excess = (1 + excess).cumprod(axis=1) * 100
    running_max = cum_excess.cummax(axis=1)
    drawdown = (cum_excess - running_max) / running_max * 100
    max_dd = drawdown.min(axis=1)

    result = pd.DataFrame({"fund_name": fund_names, "excess_max_drawdown_pct": max_dd})

    if save_path:
        result.to_csv(save_path, index=True)

    return result


def calculate_benchmark_max_drawdown(benchmark_daily_returns):
    """Max drawdown (%) of the benchmark's own cumulative return over the same period."""
    benchmark = benchmark_daily_returns.copy()
    benchmark.index = [pd.Timestamp(idx).strftime("%Y-%m-%d") for idx in benchmark.index]
    bench_ret = benchmark.iloc[:, 0] / 100
    cum = (1 + bench_ret).cumprod() * 100
    running_max = cum.cummax()
    drawdown = (cum - running_max) / running_max * 100
    return drawdown.min()


def build_fund_scorecard(fund_nav_data, fund_daily_returns, benchmark_daily_returns,
                          trading_days=242, save_path=None):
    """
    One row per fund: annualized return/vol/Sharpe, annualized excess return,
    tracking error, and information ratio vs. the group benchmark, plus:
      - max_drawdown_pct: NAV (total-return) max drawdown
      - excess_max_drawdown_pct: max drawdown of cumulative EXCESS return -
        the metric that actually isolates the enhancement's downside risk
      - benchmark_max_drawdown_pct: the benchmark's own max drawdown, for reference
      - drawdown_ratio: max_drawdown_pct / benchmark_max_drawdown_pct (both
        negative, so this ratio is positive) - >1 means the fund's NAV drew
        down MORE than its benchmark (amplification); <1 means LESS (buffer)
    Sorted by information ratio (descending).
    """
    fund_names = fund_daily_returns["fund_name"]
    returns = fund_daily_returns.drop(columns=["fund_name"]) / 100  # back to decimal

    excess_returns_pct = calculate_excess_returns(fund_daily_returns, benchmark_daily_returns)
    excess = excess_returns_pct.drop(columns=["fund_name"]) / 100

    max_dd = calculate_max_drawdown(fund_nav_data)
    excess_max_dd = calculate_excess_max_drawdown(fund_daily_returns, benchmark_daily_returns)
    benchmark_max_dd = calculate_benchmark_max_drawdown(benchmark_daily_returns)

    ann_return = returns.mean(axis=1) * trading_days * 100
    ann_vol = returns.std(axis=1) * np.sqrt(trading_days) * 100
    sharpe = ann_return / ann_vol

    ann_excess_return = excess.mean(axis=1) * trading_days * 100
    tracking_error = excess.std(axis=1) * np.sqrt(trading_days) * 100
    information_ratio = ann_excess_return / tracking_error

    scorecard = pd.DataFrame({
        "fund_name": fund_names,
        "ann_return_pct": ann_return,
        "ann_vol_pct": ann_vol,
        "sharpe": sharpe,
        "ann_excess_return_pct": ann_excess_return,
        "tracking_error_pct": tracking_error,
        "information_ratio": information_ratio,
    }).join(max_dd["max_drawdown_pct"]).join(excess_max_dd["excess_max_drawdown_pct"])

    scorecard["benchmark_max_drawdown_pct"] = benchmark_max_dd
    scorecard["drawdown_ratio"] = scorecard["max_drawdown_pct"] / benchmark_max_dd

    scorecard = scorecard.sort_values("information_ratio", ascending=False)

    if save_path:
        scorecard.to_csv(save_path, index=True)

    return scorecard


In [ ]:
# Run the full pipeline (NAV -> daily returns -> benchmark -> scorecard) for every group.
scorecards = {}
nav_data_by_group = {}
daily_returns_by_group = {}
benchmark_returns_by_group = {}
benchmark_returns_by_index_code = {}  # groups sharing an index only fetch it once;
                                       # also reused later by the standalone active-quant
                                       # pipeline, which shares CSI500 (000905.SH) with CSI500

for group_name, group in FUND_GROUPS.items():
    fund_codes = group["fund_codes"]
    index_code = group["index_code"]

    if not fund_codes:
        continue

    nav_data = get_fund_nav_data(
        fund_codes, "2025-07-01", "2026-07-01",
        save_path=f"fund_nav_data_{group_name.lower()}.csv",
    )
    if nav_data is None:
        print(f"Skipping {group_name}: failed to fetch fund NAV data (see Wind error above).")
        continue

    if index_code in benchmark_returns_by_index_code:
        benchmark_returns = benchmark_returns_by_index_code[index_code]
    else:
        benchmark_returns = get_index_daily_returns(
            index_code, "2025-07-01", "2026-07-01",
            save_path=f"index_daily_returns_{group_name.lower()}.csv",
        )
        if benchmark_returns is not None:
            benchmark_returns_by_index_code[index_code] = benchmark_returns

    if benchmark_returns is None:
        print(f"Skipping {group_name}: failed to fetch benchmark index data (see Wind error above).")
        continue

    daily_returns = calculate_fund_daily_returns(
        nav_data, save_path=f"fund_daily_returns_{group_name.lower()}.csv",
    )

    scorecard = build_fund_scorecard(
        nav_data, daily_returns, benchmark_returns,
        save_path=f"fund_scorecard_{group_name.lower()}.csv",
    )
    scorecard.insert(0, "benchmark_group", group_name)
    scorecards[group_name] = scorecard

    # Keep the underlying series around (not just the scorecard) for the
    # cross-group and top-performer deep-dive sections below.
    nav_data_by_group[group_name] = nav_data
    daily_returns_by_group[group_name] = daily_returns
    benchmark_returns_by_group[group_name] = benchmark_returns

master_scorecard = pd.concat(scorecards.values(), axis=0)
master_scorecard.to_csv("fund_scorecard_all.csv", index=True)
print(master_scorecard)


## Visualize the scorecard

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
import os

# Pick the first installed font that supports CJK characters, so fund names
# (in Chinese) render instead of showing up as missing-glyph boxes.
_CJK_FONT_CANDIDATES = [
    "PingFang SC", "Heiti SC", "STHeiti", "Arial Unicode MS",
    "Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "WenQuanYi Zen Hei",
]
_available_fonts = {f.name for f in font_manager.fontManager.ttflist}
_cjk_font = next((f for f in _CJK_FONT_CANDIDATES if f in _available_fonts), None)

# Fallback: macOS/Windows CJK fonts often ship as .ttc collection files that
# matplotlib's font cache doesn't always index by family name, so the name-based
# lookup above can miss a font that is actually installed. Register known font
# file paths directly instead.
if _cjk_font is None:
    _CJK_FONT_PATHS = [
        "/System/Library/Fonts/PingFang.ttc",
        "/System/Library/Fonts/STHeiti Light.ttc",
        "/System/Library/Fonts/STHeiti Medium.ttc",
        "/Library/Fonts/Arial Unicode.ttf",
        "C:/Windows/Fonts/msyh.ttc",
        "C:/Windows/Fonts/simhei.ttf",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
    ]
    for _path in _CJK_FONT_PATHS:
        if os.path.exists(_path):
            font_manager.fontManager.addfont(_path)
            _cjk_font = font_manager.FontProperties(fname=_path).get_name()
            break

if _cjk_font:
    plt.rcParams["font.sans-serif"] = [_cjk_font]
    plt.rcParams["font.family"] = "sans-serif"
else:
    print(
        "No CJK font found by name or file path - Chinese fund names will still "
        "show as boxes. Install a CJK font (e.g. Noto Sans CJK) and rerun this cell."
    )

plt.rcParams["axes.unicode_minus"] = False  # avoid garbled minus signs with CJK fonts
print("Using font:", _cjk_font)


In [ ]:
# Real Chinese benchmark names for chart titles/legends/axis labels, instead
# of the internal FUND_GROUPS keys.
GROUP_DISPLAY_NAMES = {
    "CSI500": "中证500",
    "CSI800": "中证800",
    "CSI1000": "中证1000",
    "CSI2000": "中证2000",
    "CNI2000": "国证2000",
    "CSI500_ActiveQuant": "中证500 (主动量化)",
}


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

groups = master_scorecard["benchmark_group"].unique()
colors = plt.cm.tab10.colors

for i, group in enumerate(groups):
    subset = master_scorecard[master_scorecard["benchmark_group"] == group]
    ax.scatter(
        subset["tracking_error_pct"],
        subset["information_ratio"],
        s=subset["ann_vol_pct"] * 3,  # bubble size ~ annualized volatility
        alpha=0.7,
        color=colors[i % len(colors)],
        label=GROUP_DISPLAY_NAMES.get(group, group),
    )

ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax.set_xlabel("Tracking Error (%)")
ax.set_ylabel("Information Ratio")
ax.set_title("Information Ratio vs. Tracking Error (bubble size = annualized volatility)")
ax.legend(title="Benchmark Group")

plt.tight_layout()
plt.savefig("fund_ir_vs_te_scatter.png", dpi=150)
plt.show()

In [ ]:
top15 = master_scorecard.sort_values("information_ratio", ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top15["fund_name"], top15["information_ratio"], color="steelblue")
ax.invert_yaxis()
ax.set_xlabel("Information Ratio")
ax.set_title("Top 15 Funds by Information Ratio (across all benchmark groups)")
plt.tight_layout()
plt.savefig("fund_top15_ir_bar.png", dpi=150)
plt.show()

## Cross-benchmark-group comparison

In [ ]:
group_summary = master_scorecard.groupby("benchmark_group")[
    ["ann_return_pct", "ann_vol_pct", "sharpe", "ann_excess_return_pct",
     "tracking_error_pct", "information_ratio", "max_drawdown_pct",
     "excess_max_drawdown_pct", "drawdown_ratio"]
].agg(["mean", "median", "std"])

print(group_summary)


In [ ]:
group_order = [g for g in FUND_GROUPS if g in daily_returns_by_group]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ir_by_group = [
    master_scorecard.loc[master_scorecard["benchmark_group"] == g, "information_ratio"].dropna()
    for g in group_order
]
excess_by_group = [
    master_scorecard.loc[master_scorecard["benchmark_group"] == g, "ann_excess_return_pct"].dropna()
    for g in group_order
]

axes[0].boxplot(ir_by_group, labels=[GROUP_DISPLAY_NAMES.get(g, g) for g in group_order])
axes[0].axhline(0, color="gray", linestyle="--", linewidth=0.8)
axes[0].set_title("Information Ratio Distribution by Benchmark Group")
axes[0].set_ylabel("Information Ratio")

axes[1].boxplot(excess_by_group, labels=[GROUP_DISPLAY_NAMES.get(g, g) for g in group_order])
axes[1].axhline(0, color="gray", linestyle="--", linewidth=0.8)
axes[1].set_title("Annualized Excess Return Distribution by Benchmark Group")
axes[1].set_ylabel("Annualized Excess Return (%)")

plt.tight_layout()
plt.savefig("group_comparison_boxplots.png", dpi=150)
plt.show()


## Drawdown analysis: excess drawdown, amplification ratio, and cross-group comparison

NAV (total-return) max drawdown mostly reflects the benchmark's own drawdown for a ~beta-1 enhanced fund - if the market falls 14%, everyone falls ~14%, which measures beta, not the enhancement. Three views instead: (1) max drawdown of **excess return** (fund vs. its own benchmark) - the downside-risk metric that actually isolates the enhancement itself; (2) the **drawdown ratio** (fund NAV drawdown / benchmark drawdown) - whether the fund amplified or buffered the benchmark's own worst decline; (3) both compared **across benchmark groups**.

In [ ]:
pd.set_option("display.max_rows", None)

dd_cols = ["fund_name", "benchmark_group", "max_drawdown_pct", "excess_max_drawdown_pct",
           "benchmark_max_drawdown_pct", "drawdown_ratio"]

print("Worst excess-return drawdowns (the metric that matters for an enhanced product):")
print(master_scorecard.sort_values("excess_max_drawdown_pct")[dd_cols].head(20))

pd.reset_option("display.max_rows")


In [ ]:
pd.set_option("display.max_rows", None)

print("Drawdown ratio (fund NAV drawdown / benchmark drawdown) - >1 = amplified, <1 = buffered:")
print(master_scorecard.sort_values("drawdown_ratio", ascending=False)[dd_cols])

pd.reset_option("display.max_rows")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

metrics = [
    ("max_drawdown_pct", "NAV Max Drawdown (%)"),
    ("excess_max_drawdown_pct", "Excess-Return Max Drawdown (%)"),
    ("drawdown_ratio", "Drawdown Ratio (fund / benchmark)"),
]

for ax, (col, title) in zip(axes, metrics):
    data_by_group = [
        master_scorecard.loc[master_scorecard["benchmark_group"] == g, col].dropna()
        for g in group_order
    ]
    ax.boxplot(data_by_group, labels=[GROUP_DISPLAY_NAMES.get(g, g) for g in group_order])
    if col == "drawdown_ratio":
        ax.axhline(1, color="gray", linewidth=0.8, linestyle="--")
    else:
        ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
    ax.set_title(title)

plt.tight_layout()
plt.savefig("drawdown_comparison_by_group.png", dpi=150)
plt.show()


## Top-performer deep dive

In [ ]:
def cumulative_return_series(fund_code, group_name):
    """Cumulative return (indexed to 100) for a fund and its group benchmark, aligned by date."""
    daily_returns = daily_returns_by_group[group_name]
    benchmark_returns = benchmark_returns_by_group[group_name]

    fund_name = daily_returns.loc[fund_code, "fund_name"]
    fund_ret_full = daily_returns.loc[fund_code].drop("fund_name").astype(float) / 100

    # Restrict to the fund's own actual history. Funds that launched partway
    # through the window have NaN before their inception - starting from their
    # first valid date means both lines get rebased to 100 at the same point,
    # instead of the benchmark carrying a "head start" from before the fund
    # existed, which would make the visual gap overstate relative performance.
    first_valid = fund_ret_full.first_valid_index()
    fund_ret = fund_ret_full.loc[first_valid:]

    benchmark = benchmark_returns.copy()
    benchmark.index = [pd.Timestamp(idx).strftime("%Y-%m-%d") for idx in benchmark.index]
    bench_ret = benchmark.iloc[:, 0].reindex(fund_ret.index) / 100

    fund_cum = (1 + fund_ret).cumprod() * 100
    bench_cum = (1 + bench_ret).cumprod() * 100

    return fund_name, fund_cum, bench_cum


In [ ]:
top5 = master_scorecard.sort_values("information_ratio", ascending=False).head(5)

fig, axes = plt.subplots(len(top5), 1, figsize=(10, 4 * len(top5)))

for ax, (fund_code, row) in zip(axes, top5.iterrows()):
    fund_name, fund_cum, bench_cum = cumulative_return_series(fund_code, row["benchmark_group"])
    x = pd.to_datetime(fund_cum.index, format="%Y-%m-%d")

    ax.plot(x, fund_cum.values, label=f"{fund_name} ({fund_code})")
    bench_label = GROUP_DISPLAY_NAMES.get(row["benchmark_group"], row["benchmark_group"])
    ax.plot(x, bench_cum.values, label=f"{bench_label} benchmark", linestyle="--", color="gray")
    ax.set_title(f"{fund_name} ({fund_code}) vs {bench_label} benchmark - IR={row['information_ratio']:.2f}")
    ax.set_ylabel("Cumulative Return (indexed to 100)")
    ax.legend()

plt.tight_layout()
plt.savefig("top5_cumulative_vs_benchmark.png", dpi=150)
plt.show()


In [ ]:
def flag_outlier_days(fund_code, group_name, z_thresh=3):
    """Fund daily returns more than z_thresh standard deviations from the fund's own mean."""
    daily_returns = daily_returns_by_group[group_name]
    ret = daily_returns.loc[fund_code].drop("fund_name").astype(float)
    z = (ret - ret.mean()) / ret.std()
    return ret[z.abs() > z_thresh]


for fund_code, row in top5.iterrows():
    outliers = flag_outlier_days(fund_code, row["benchmark_group"])
    if len(outliers):
        print(f"{row['fund_name']} ({fund_code}) - outlier days (>3 sigma from its own mean):")
        print(outliers)
        print()


In [ ]:
def split_period_ir(daily_returns, benchmark_daily_returns, trading_days_per_year=242):
    """
    Information ratio computed separately for the first and second half of the
    period. A fund with a strong IR in only one half is more likely a lucky
    stretch than a fund whose IR holds up in both halves.
    """
    fund_names = daily_returns["fund_name"]
    excess_pct = calculate_excess_returns(daily_returns, benchmark_daily_returns)
    excess = excess_pct.drop(columns=["fund_name"]) / 100

    mid = excess.shape[1] // 2
    halves = {"H1": excess.iloc[:, :mid], "H2": excess.iloc[:, mid:]}

    result = pd.DataFrame({"fund_name": fund_names})
    for label, sub in halves.items():
        ann_excess = sub.mean(axis=1) * trading_days_per_year * 100
        tracking_error = sub.std(axis=1) * np.sqrt(trading_days_per_year) * 100
        result[f"ir_{label}"] = ann_excess / tracking_error

    result["ir_persistence"] = result[["ir_H1", "ir_H2"]].min(axis=1)
    return result


persistence_frames = []
for group_name in daily_returns_by_group:
    p = split_period_ir(daily_returns_by_group[group_name], benchmark_returns_by_group[group_name])
    p.insert(0, "benchmark_group", group_name)
    persistence_frames.append(p)

persistence_table = pd.concat(persistence_frames, axis=0).sort_values("ir_persistence", ascending=False)
persistence_table.to_csv("fund_ir_persistence.csv", index=True)
print(persistence_table.head(15))


## Worst-performer deep dive

In [ ]:
bottom5 = master_scorecard.sort_values("information_ratio", ascending=True).head(5)

fig, axes = plt.subplots(len(bottom5), 1, figsize=(10, 4 * len(bottom5)))

for ax, (fund_code, row) in zip(axes, bottom5.iterrows()):
    fund_name, fund_cum, bench_cum = cumulative_return_series(fund_code, row["benchmark_group"])
    x = pd.to_datetime(fund_cum.index, format="%Y-%m-%d")

    ax.plot(x, fund_cum.values, label=f"{fund_name} ({fund_code})", color="firebrick")
    bench_label = GROUP_DISPLAY_NAMES.get(row["benchmark_group"], row["benchmark_group"])
    ax.plot(x, bench_cum.values, label=f"{bench_label} benchmark", linestyle="--", color="gray")
    ax.set_title(f"{fund_name} ({fund_code}) vs {bench_label} benchmark - IR={row['information_ratio']:.2f}")
    ax.set_ylabel("Cumulative Return (indexed to 100)")
    ax.legend()

plt.tight_layout()
plt.savefig("bottom5_cumulative_vs_benchmark.png", dpi=150)
plt.show()


In [ ]:
for fund_code, row in bottom5.iterrows():
    outliers = flag_outlier_days(fund_code, row["benchmark_group"])
    if len(outliers):
        print(f"{row['fund_name']} ({fund_code}) - outlier days (>3 sigma from its own mean):")
        print(outliers)
        print()


## Within-group ranking

In [ ]:
# Rank and percentile within each fund's own benchmark group - comparing IR
# straight across groups isn't fully fair, since some benchmarks are
# structurally easier to generate alpha against than others (see the
# cross-group comparison above).
master_scorecard["rank_in_group"] = (
    master_scorecard.groupby("benchmark_group")["information_ratio"]
    .rank(ascending=False, method="min")
    .astype(int)
)
master_scorecard["group_size"] = master_scorecard.groupby("benchmark_group")["information_ratio"].transform("size")
master_scorecard["percentile_in_group"] = 1 - (master_scorecard["rank_in_group"] - 1) / (
    (master_scorecard["group_size"] - 1).clip(lower=1)
)

master_scorecard.to_csv("fund_scorecard_all.csv", index=True)

display_cols = ["fund_name", "information_ratio", "ann_excess_return_pct", "max_drawdown_pct", "rank_in_group"]

for group_name in FUND_GROUPS:
    if group_name not in daily_returns_by_group:
        continue
    group_df = master_scorecard[master_scorecard["benchmark_group"] == group_name].sort_values("rank_in_group")
    print(f"=== {group_name}: best 3 (of {group_df.shape[0]}) ===")
    print(group_df.head(3)[display_cols])
    print(f"=== {group_name}: worst 3 ===")
    print(group_df.tail(3)[display_cols])
    print()


## Selecting the best funds to invest in (per benchmark group)

Not a fixed top-N - a data-driven bar. A fund must clear `information_ratio > 1.0` **and** show positive IR in both halves of the year (real consistency, not a lucky stretch), then the best by IR up to 5 are kept. A group can yield anywhere from 0 to 5 funds depending on how many genuinely clear the bar.

In [ ]:
def select_best_funds(scorecard, persistence, min_ir=1.0, max_picks=5, require_drawdown_better_than_median=True):
    """
    For each benchmark group, select funds worth investing in. A fund must:
      1. Clear a minimum information ratio bar (min_ir)
      2. Show positive IR persistence (positive IR in every sub-period of the
         `persistence` table - both halves for the 1-year check, all three
         fiscal years for the 3-year check)
      3. (if require_drawdown_better_than_median) Have an excess-return max
         drawdown no worse than its group's median - screens out funds whose
         average IR looks good only because it survived one brutal
         excess-drawdown stretch that a period-average IR doesn't penalize.
    Ranked by IR among survivors. Not a fixed count per group.
    """
    persistence_cols = [c for c in persistence.columns if c not in ("benchmark_group", "fund_name")]
    universe = scorecard.join(persistence[persistence_cols])

    selections = {}
    for group_name in scorecard["benchmark_group"].unique():
        group_universe = universe[universe["benchmark_group"] == group_name]

        candidates = group_universe[
            (group_universe["information_ratio"] > min_ir) &
            (group_universe["ir_persistence"] > 0)
        ]

        if require_drawdown_better_than_median:
            median_dd = group_universe["excess_max_drawdown_pct"].median()
            candidates = candidates[candidates["excess_max_drawdown_pct"] >= median_dd]

        candidates = candidates.sort_values("information_ratio", ascending=False)
        selections[group_name] = candidates.head(max_picks)

    return selections


MIN_IR_BAR = 1.0
MAX_PICKS = 5

best_funds_by_group = select_best_funds(master_scorecard, persistence_table, MIN_IR_BAR, MAX_PICKS)

display_cols = ["fund_name", "information_ratio", "ann_excess_return_pct",
                "excess_max_drawdown_pct", "drawdown_ratio", "ir_H1", "ir_H2"]

for group_name, picks in best_funds_by_group.items():
    display_name = GROUP_DISPLAY_NAMES.get(group_name, group_name)
    print(f"=== {display_name}: {len(picks)} fund(s) selected (IR > {MIN_IR_BAR}, positive in both halves, drawdown <= group median) ===")
    if len(picks):
        print(picks[display_cols])
    else:
        print("No fund in this group clears the bar.")
    print()


In [ ]:
best_funds_table = pd.concat(
    [picks.assign(benchmark_group_display=GROUP_DISPLAY_NAMES.get(g, g))
     for g, picks in best_funds_by_group.items() if len(picks)],
    axis=0,
) if any(len(p) for p in best_funds_by_group.values()) else pd.DataFrame()

if best_funds_table.empty:
    print("No fund in any group clears the selection bar.")
else:
    best_funds_table = best_funds_table.sort_values(
        ["benchmark_group_display", "information_ratio"], ascending=[True, False]
    )
    pd.set_option("display.max_rows", None)
    print(best_funds_table[[
        "benchmark_group_display", "fund_name", "information_ratio",
        "ann_excess_return_pct", "excess_max_drawdown_pct", "drawdown_ratio",
    ]])
    pd.reset_option("display.max_rows")


In [ ]:
from IPython.display import display, Markdown

table_display_cols = ["fund_name", "information_ratio", "ann_excess_return_pct",
                       "excess_max_drawdown_pct", "drawdown_ratio", "ir_H1", "ir_H2"]

for group_name, picks in best_funds_by_group.items():
    display_name = GROUP_DISPLAY_NAMES.get(group_name, group_name)
    display(Markdown(f"**{display_name}: recommended funds** ({len(picks)} selected)"))
    if len(picks):
        display(picks[table_display_cols])
    else:
        display(Markdown("_No fund in this group clears the selection bar._"))


In [ ]:
import matplotlib.patches as mpatches

if not best_funds_table.empty:
    labels = [
        f"{row.fund_name} ({row.benchmark_group_display})"
        for row in best_funds_table.itertuples()
    ]
    unique_groups = list(best_funds_table["benchmark_group_display"].unique())
    colors_map = {g: c for g, c in zip(unique_groups, plt.cm.tab10.colors)}
    bar_colors = [colors_map[g] for g in best_funds_table["benchmark_group_display"]]

    fig, axes = plt.subplots(3, 1, figsize=(10, max(3, 0.4 * len(best_funds_table)) * 3))

    panels = [
        ("information_ratio", "Information Ratio"),
        ("ann_excess_return_pct", "Annualized Excess Return (%)"),
        ("excess_max_drawdown_pct", "Excess-Return Max Drawdown (%)"),
    ]

    for ax, (col, title) in zip(axes, panels):
        ax.barh(labels, best_funds_table[col], color=bar_colors)
        ax.invert_yaxis()
        ax.axvline(0, color="gray", linewidth=0.8)
        ax.set_title(f"Selected Best Funds: {title}")

    legend_handles = [mpatches.Patch(color=c, label=g) for g, c in colors_map.items()]
    axes[0].legend(handles=legend_handles, loc="lower right", fontsize=8)

    plt.tight_layout()
    plt.savefig("best_funds_selected_charts.png", dpi=150)
    plt.show()


## Risk vs. reward: is extra tracking error actually rewarded?

In [ ]:
risk_reward_corr = (
    master_scorecard.groupby("benchmark_group")
    .apply(lambda df: df["tracking_error_pct"].corr(df["ann_excess_return_pct"]))
    .rename("corr_tracking_error_vs_excess_return")
)

print(risk_reward_corr)


In [ ]:
fig, axes = plt.subplots(1, len(group_order), figsize=(5 * len(group_order), 4), sharey=True)

for ax, group_name in zip(axes, group_order):
    subset = master_scorecard[master_scorecard["benchmark_group"] == group_name]
    ax.scatter(subset["tracking_error_pct"], subset["ann_excess_return_pct"], alpha=0.7)

    if len(subset) >= 2:
        coeffs = np.polyfit(subset["tracking_error_pct"], subset["ann_excess_return_pct"], 1)
        trend_x = np.linspace(subset["tracking_error_pct"].min(), subset["tracking_error_pct"].max(), 10)
        ax.plot(trend_x, np.polyval(coeffs, trend_x), color="firebrick", linestyle="--")

    ax.axhline(0, color="gray", linewidth=0.8, linestyle=":")
    corr = subset["tracking_error_pct"].corr(subset["ann_excess_return_pct"])
    ax.set_title(f"{GROUP_DISPLAY_NAMES.get(group_name, group_name)} (corr={corr:.2f})")
    ax.set_xlabel("Tracking Error (%)")

axes[0].set_ylabel("Annualized Excess Return (%)")
plt.tight_layout()
plt.savefig("risk_vs_reward_by_group.png", dpi=150)
plt.show()


## Active-quant (主动量化) fund group

58 actively-managed quant funds benchmarked against CSI500 (000905.SH), distinct from the index-enhanced CSI500 group above.

In [ ]:
ACTIVE_QUANT_GROUP_NAME = "CSI500_ActiveQuant"
ACTIVE_QUANT_INDEX_CODE = "000905.SH"  # same CSI500 index as the index-enhanced CSI500 group above
ACTIVE_QUANT_START_DATE = "2025-07-01"
ACTIVE_QUANT_END_DATE = "2026-07-01"

active_quant_nav_data = get_fund_nav_data(
    fund_codes_ACTIVE_QUANT_CSI500, ACTIVE_QUANT_START_DATE, ACTIVE_QUANT_END_DATE,
    save_path=f"fund_nav_data_{ACTIVE_QUANT_GROUP_NAME.lower()}.csv",
)

if active_quant_nav_data is None:
    print(f"Failed to fetch NAV data for {ACTIVE_QUANT_GROUP_NAME} (see Wind error above).")
else:
    active_quant_daily_returns = calculate_fund_daily_returns(
        active_quant_nav_data, save_path=f"fund_daily_returns_{ACTIVE_QUANT_GROUP_NAME.lower()}.csv",
    )

    # Reuse the CSI500 benchmark already fetched for the index-enhanced CSI500
    # group above (same index, same date range) instead of hitting Wind again.
    if ACTIVE_QUANT_INDEX_CODE in benchmark_returns_by_index_code:
        active_quant_benchmark_returns = benchmark_returns_by_index_code[ACTIVE_QUANT_INDEX_CODE]
    else:
        active_quant_benchmark_returns = get_index_daily_returns(
            ACTIVE_QUANT_INDEX_CODE, ACTIVE_QUANT_START_DATE, ACTIVE_QUANT_END_DATE,
            save_path=f"index_daily_returns_{ACTIVE_QUANT_GROUP_NAME.lower()}.csv",
        )

    active_quant_scorecard = build_fund_scorecard(
        active_quant_nav_data, active_quant_daily_returns, active_quant_benchmark_returns,
        save_path=f"fund_scorecard_{ACTIVE_QUANT_GROUP_NAME.lower()}.csv",
    )
    active_quant_scorecard.insert(0, "benchmark_group", ACTIVE_QUANT_GROUP_NAME)

    # Populate the same shared per-group dicts the index-enhanced pipeline uses,
    # so cumulative_return_series / flag_outlier_days / group_average_cumulative_return
    # work unchanged for this group too - without ever adding it to FUND_GROUPS or
    # master_scorecard, so it stays out of the index-enhanced-only sections above.
    nav_data_by_group[ACTIVE_QUANT_GROUP_NAME] = active_quant_nav_data
    daily_returns_by_group[ACTIVE_QUANT_GROUP_NAME] = active_quant_daily_returns
    benchmark_returns_by_group[ACTIVE_QUANT_GROUP_NAME] = active_quant_benchmark_returns

    # Fund code and name for every active-quant fund in this group.
    print(active_quant_scorecard[["fund_name"]])


In [ ]:
pd.set_option("display.max_rows", None)

active_quant_ranked = active_quant_scorecard.sort_values("information_ratio", ascending=False)
print(active_quant_ranked[[
    "fund_name", "ann_return_pct", "ann_vol_pct", "sharpe",
    "ann_excess_return_pct", "tracking_error_pct", "information_ratio", "max_drawdown_pct",
]])

pd.reset_option("display.max_rows")


## Active-quant top 5 (within-group) vs CSI500

In [ ]:
quant_top5 = active_quant_scorecard.sort_values("information_ratio", ascending=False).head(5)

fig, axes = plt.subplots(len(quant_top5), 1, figsize=(10, 4 * len(quant_top5)))

for ax, (fund_code, row) in zip(axes, quant_top5.iterrows()):
    fund_name, fund_cum, bench_cum = cumulative_return_series(fund_code, row["benchmark_group"])
    x = pd.to_datetime(fund_cum.index, format="%Y-%m-%d")

    ax.plot(x, fund_cum.values, label=f"{fund_name} ({fund_code})")
    ax.plot(x, bench_cum.values, label="中证500 benchmark", linestyle="--", color="gray")
    ax.set_title(f"{fund_name} ({fund_code}) vs 中证500 - IR={row['information_ratio']:.2f}")
    ax.set_ylabel("Cumulative Return (indexed to 100)")
    ax.legend()

plt.tight_layout()
plt.savefig("active_quant_top5_vs_benchmark.png", dpi=150)
plt.show()


## Category comparison: active-quant vs. index-enhanced, both vs. CSI500

Equal-weighted, daily-rebalanced average of each category vs. the shared 000905.SH benchmark - restricted to funds with a complete July 2025-July 2026 history (excludes recently-launched funds) so the average isn't distorted by composition drift from funds joining partway through.

In [ ]:
def group_average_cumulative_return(group_name, survivors_only=True):
    """
    Cumulative return of a hypothetical daily-rebalanced equal-weight portfolio of
    the group. If survivors_only, restricts to funds with a complete return
    history for the full period - avoids composition drift from funds that
    launched partway through the window joining the average partway through.
    """
    daily_returns = daily_returns_by_group[group_name]
    returns = daily_returns.drop(columns=["fund_name"]).astype(float) / 100

    if survivors_only:
        n_total = returns.shape[0]
        returns = returns.dropna(axis=0, how="any")
        n_survivors = returns.shape[0]
        print(f"{group_name}: using {n_survivors} of {n_total} funds with a complete history")

    avg_daily_return = returns.mean(axis=0)
    return (1 + avg_daily_return).cumprod() * 100


def benchmark_cumulative_return(group_name):
    benchmark_returns = benchmark_returns_by_group[group_name].copy()
    benchmark_returns.index = [pd.Timestamp(idx).strftime("%Y-%m-%d") for idx in benchmark_returns.index]
    bench_ret = benchmark_returns.iloc[:, 0] / 100
    return (1 + bench_ret).cumprod() * 100


active_quant_avg = group_average_cumulative_return("CSI500_ActiveQuant")
index_enhanced_avg = group_average_cumulative_return("CSI500")
csi500_cum = benchmark_cumulative_return("CSI500")  # same 000905.SH index both groups benchmark against

fig, ax = plt.subplots(figsize=(11, 6))
x = pd.to_datetime(csi500_cum.index, format="%Y-%m-%d")

ax.plot(x, csi500_cum.values, label="中证500 benchmark", color="gray", linestyle="--")
ax.plot(x, index_enhanced_avg.reindex(csi500_cum.index).values, label="Index-enhanced 中证500 (equal-weight avg, survivors only)")
ax.plot(x, active_quant_avg.reindex(csi500_cum.index).values, label="Active-quant 中证500 (equal-weight avg, survivors only)")

ax.set_ylabel("Cumulative Return (indexed to 100)")
ax.set_title("Category comparison vs 中证500: index-enhanced vs active-quant (funds with full-period history only)")
ax.legend()
plt.tight_layout()
plt.savefig("category_comparison_vs_csi500.png", dpi=150)
plt.show()


### Is the equal-weight average being distorted, or is underperformance broad-based?

Before spending Wind quota fetching AUM data to build an AUM-weighted comparison, check for free whether it would even matter: if mean and median are close together and the spread is tight, most funds are similarly weak and AUM-weighting won't rescue the picture - a few big funds being better wouldn't be enough to offset a broad-based problem. If mean is much more negative than median (left-skewed), a handful of very bad funds are dragging the equal-weight average down, and AUM-weighting could tell a meaningfully different story.

In [ ]:
print("Active-quant CSI500 (CSI500_ActiveQuant) ann_excess_return_pct distribution:")
print(active_quant_scorecard["ann_excess_return_pct"].describe())

print()
print("Index-enhanced CSI500 ann_excess_return_pct distribution (for comparison):")
print(master_scorecard.loc[master_scorecard["benchmark_group"] == "CSI500", "ann_excess_return_pct"].describe())


## Verify: are the top/worst funds' outlier days the same market-wide events?

Tests two specific claims: (1) outlier days cluster on the same dates for both winners and losers (market-wide event, not fund-specific), and (2) winners' down-day outliers get offset by up-day outliers while losers only show one-sided down-day outliers.

In [ ]:
def collect_outlier_days(fund_scorecard_subset, label):
    """Tidy (fund, date, return) table of every outlier day flagged for a set of funds."""
    rows = []
    for fund_code, row in fund_scorecard_subset.iterrows():
        outliers = flag_outlier_days(fund_code, row["benchmark_group"])
        for date, ret in outliers.items():
            rows.append({
                "fund_code": fund_code,
                "fund_name": row["fund_name"],
                "group": label,
                "date": date,
                "return_pct": ret,
            })
    return pd.DataFrame(rows)


winner_outliers = collect_outlier_days(top5, "winner")
loser_outliers = collect_outlier_days(bottom5, "loser")
all_outliers = pd.concat([winner_outliers, loser_outliers], ignore_index=True)

print(all_outliers.sort_values(["date", "group", "fund_code"]).to_string(index=False))


In [ ]:
# Claim to test #1: are outlier days shared market-wide events (same handful
# of dates hit both winners and losers), or fund-specific idiosyncratic moves?
date_counts = all_outliers.groupby("date").size().sort_values(ascending=False)
print("Outlier days ranked by how many (winner+loser) funds were flagged on that date:")
print(date_counts.head(10))


In [ ]:
# Claim to test #2: do winners have offsetting up-days for their down-days,
# while losers only show one-sided down-day outliers with no matching up-days?
sign_summary = (
    all_outliers.groupby(["group", "fund_code", "fund_name"])["return_pct"]
    .apply(lambda s: pd.Series({"n_up_outliers": (s > 0).sum(), "n_down_outliers": (s < 0).sum()}))
    .unstack()
)
sign_summary["net"] = sign_summary["n_up_outliers"] - sign_summary["n_down_outliers"]
print(sign_summary.sort_values(["group", "net"]))


## Better dates: benchmark-defined events + full-universe cross-check

The date list above came from only 10 hand-picked funds (top-5/bottom-5) -
a small, biased sample. This defines "market event" directly from each
benchmark's own extreme daily moves (sample-independent), then checks how
many funds across the *entire* universe (~124 funds, not 10) were flagged
on the same five candidate dates.

In [ ]:
def top_benchmark_extreme_days(index_code, n=10):
    """Days the benchmark itself moved the most, ranked by absolute daily return -
    defines "market event" directly from the index, independent of any fund sample."""
    benchmark_returns = benchmark_returns_by_index_code[index_code]
    bench = benchmark_returns.copy()
    bench.index = [pd.Timestamp(idx).strftime("%Y-%m-%d") for idx in bench.index]
    ret = bench.iloc[:, 0]
    return ret.reindex(ret.abs().sort_values(ascending=False).index).head(n)


for group_name, group in FUND_GROUPS.items():
    display_name = GROUP_DISPLAY_NAMES.get(group_name, group_name)
    print(f"=== {display_name} ({group['index_code']}) - top 10 extreme benchmark days ===")
    print(top_benchmark_extreme_days(group["index_code"]))
    print()


In [ ]:
def funds_flagged_on_date(date, z_thresh=3):
    """Across every index-enhanced fund (not just the top/bottom 5, and excluding
    the separately-tracked active-quant group), which ones had a >z_thresh sigma
    move (vs. their own mean/std) on this specific date."""
    hits = []
    for group_name in FUND_GROUPS:
        daily_returns = daily_returns_by_group[group_name]
        returns = daily_returns.drop(columns=["fund_name"]).astype(float)
        if date not in returns.columns:
            continue
        means = returns.mean(axis=1)
        stds = returns.std(axis=1)
        z = (returns[date] - means) / stds
        flagged = z[z.abs() > z_thresh]
        for fund_code, zscore in flagged.items():
            hits.append({
                "group": group_name,
                "fund_code": fund_code,
                "fund_name": daily_returns.loc[fund_code, "fund_name"],
                "return_pct": returns.loc[fund_code, date],
                "zscore": zscore,
            })
    return pd.DataFrame(hits)


candidate_dates = ["2026-02-02", "2026-03-03", "2026-03-23", "2026-04-08", "2026-06-15"]

for date in candidate_dates:
    hits = funds_flagged_on_date(date)
    total_funds = sum(daily_returns_by_group[g].shape[0] for g in FUND_GROUPS)
    print(f"{date}: {len(hits)} of {total_funds} funds (across the full universe) flagged")

print()
print("Full detail for the first candidate date (adjust index to inspect others):")
print(funds_flagged_on_date(candidate_dates[0]).sort_values("zscore"))


## Excess-return outlier days (isolating active bets from market-wide moves)

Everything above is on a total-return basis - both the fund side (`flag_outlier_days` z-scores the fund's own raw NAV return) and the benchmark side (`top_benchmark_extreme_days` uses raw index PCT_CHG). Since these funds are ~beta-1 trackers, total-return outlier days mechanically coincide with benchmark-extreme days - that's mostly testing the market, not the fund. This instead z-scores each fund's **excess return vs. its own benchmark**, which isolates days where the fund's active/idiosyncratic bets moved unusually, independent of whatever the broad market did that day.

In [ ]:
def all_excess_outlier_hits(z_thresh=3):
    """
    Every (fund, date) pair where the fund's daily EXCESS return (vs. its own
    benchmark, not total/raw return) is more than z_thresh sigma from that
    fund's own mean excess return. This isolates active/idiosyncratic bet
    blowups from broad market (beta-driven) moves - a fund can have a huge
    total-return day just because the market moved, with an unremarkable
    excess return, or vice versa: an unremarkable total-return day that hides
    a large idiosyncratic bet against/for the benchmark.
    """
    rows = []
    for group_name in FUND_GROUPS:
        daily_returns = daily_returns_by_group[group_name]
        benchmark_returns = benchmark_returns_by_group[group_name]
        excess = calculate_excess_returns(daily_returns, benchmark_returns).drop(columns=["fund_name"]).astype(float)

        means = excess.mean(axis=1)
        stds = excess.std(axis=1)
        z = excess.sub(means, axis=0).div(stds, axis=0)
        flagged = z.abs() > z_thresh

        for fund_code in excess.index:
            fund_flags = flagged.loc[fund_code]
            for date in excess.columns[fund_flags]:
                rows.append({
                    "group": group_name,
                    "fund_code": fund_code,
                    "fund_name": daily_returns.loc[fund_code, "fund_name"],
                    "date": date,
                    "excess_return_pct": excess.loc[fund_code, date],
                    "zscore": z.loc[fund_code, date],
                })
    return pd.DataFrame(rows)


excess_outlier_hits = all_excess_outlier_hits()
print(f"{len(excess_outlier_hits)} total excess-return outlier (fund, date) hits across the index-enhanced universe")


In [ ]:
excess_date_counts = excess_outlier_hits.groupby("date").size().sort_values(ascending=False)

print("Top excess-return outlier days (ranked by how many funds flagged):")
print(excess_date_counts.head(15))

print()
print("For comparison, the total-return outlier day ranking from earlier:")
print(date_counts.head(15))


## 3-year performance comparison (2023-07-01 to 2026-07-01)

**Quota-saving revision:** instead of re-fetching all 3 years, this only fetches the missing **gap period** (2023-07-01 to 2025-06-30), reusing the already-cached 1-year data for the rest.

**Incremental chunking:** even the ~2-year gap turned out to exceed a fresh account's quota in one shot, so the gap fetch is further split into small **monthly chunks per group**, each cached to its own CSV as soon as it succeeds. If Wind fails partway through (quota exceeded), the cell stops cleanly instead of crashing - already-completed chunks stay cached, and simply rerunning the pipeline cell later (after quota resets, even on a different day or account) picks up automatically from the first not-yet-fetched chunk. Run `wind_health_check()` first each time to confirm Wind is reachable before starting.

If even monthly chunks exceed quota, `CHUNK_MONTHS` can be left at 1 (the smallest clean month-based unit) while also batching `fund_codes` into smaller groups per call (see `CHUNK_MAX_FUNDS` note in the cell below) for an additional lever.

Only funds whose inception predates `2023-07-01` (a genuine complete 3-year history, not a partial one) are eligible for the 3-year best-fund selection - flagged via `has_full_3y_history`.

In [ ]:
def interpret_wind_error(error_code, message_df=None):
    """
    Classify a WindPy error code into a clear category instead of eyeballing
    raw OUTMESSAGE text. Known codes observed in this project:
      -40522017  "CWSDService: quota exceeded"   -> QUOTA (wait for reset / switch account)
      -40521007  "WSD: SkyClient request failed" -> CONNECTIVITY (network issue, NOT quota)
    Anything else is reported as UNKNOWN with the raw message for inspection.
    """
    message = ""
    if message_df is not None:
        try:
            message = str(message_df.iloc[0, 0])
        except Exception:
            message = str(message_df)

    if error_code == 0:
        category, explanation = "OK", "No error."
    elif error_code == -40522017 or "quota exceeded" in message.lower():
        category = "QUOTA"
        explanation = "Your Wind account has hit its data quota limit. Wait for it to reset, or switch accounts."
    elif error_code == -40521007 or "skyclient" in message.lower():
        category = "CONNECTIVITY"
        explanation = (
            "Network/connection failure between WindPy and Wind's servers - "
            "NOT a quota issue. Try a different network, confirm Wind "
            "Terminal shows as connected, or retry in a few minutes."
        )
    else:
        category = "UNKNOWN"
        explanation = f"Unrecognized error code. Raw message: {message!r}"

    print(f"[{category}] error_code={error_code}: {explanation}")
    return category


def wind_health_check():
    """Cheapest possible Wind request (1 fund, 1 day) to check whether the
    connection is currently working at all, without burning meaningful quota."""
    w.start()
    error_code, data = w.wsd("000001.SZ", "close", "2026-07-01", "2026-07-01", "", usedf=True)
    if error_code == 0:
        print("Wind connection is healthy right now.")
        return "OK"
    return interpret_wind_error(error_code, data)


wind_health_check()


In [ ]:
GAP_START = "2023-07-01"
GAP_END = "2025-06-30"  # day before the existing 1-year cache's start (2025-07-01) - zero overlap
CHUNK_MONTHS = 1  # size of each incremental fetch (in months)
CHUNK_MAX_FUNDS = None  # if even 1-month chunks exceed quota, also split fund_codes into batches
                        # of this size (e.g. 10) - pass fund_codes[i:i+CHUNK_MAX_FUNDS] per call


def date_chunks(start_date, end_date, chunk_months=CHUNK_MONTHS):
    """Split a date range into consecutive smaller chunks (default: monthly)."""
    chunks = []
    current_start = pd.Timestamp(start_date)
    final_end = pd.Timestamp(end_date)
    while current_start <= final_end:
        current_end = min(
            current_start + pd.DateOffset(months=chunk_months) - pd.Timedelta(days=1),
            final_end,
        )
        chunks.append((current_start.strftime("%Y-%m-%d"), current_end.strftime("%Y-%m-%d")))
        current_start = current_end + pd.Timedelta(days=1)
    return chunks


def get_fund_nav_data_chunked(fund_codes, start_date, end_date, save_prefix,
                               chunk_months=CHUNK_MONTHS, force_refresh=False):
    """
    Fetches NAV data for a date range in small time chunks (default: monthly),
    each cached to its own CSV. If Wind fails partway through (quota
    exceeded), already-fetched chunks stay cached and this returns None
    cleanly - rerun later (e.g. after quota resets, or on a different day)
    and it picks up automatically from the first not-yet-cached chunk.
    """
    chunks = date_chunks(start_date, end_date, chunk_months)
    chunk_frames = []

    for i, (chunk_start, chunk_end) in enumerate(chunks):
        chunk_path = f"{save_prefix}_chunk{i:03d}_{chunk_start}_{chunk_end}.csv"

        if not force_refresh and os.path.exists(chunk_path):
            print(f"  chunk {i + 1}/{len(chunks)} ({chunk_start} to {chunk_end}): loading from cache")
            chunk_data = pd.read_csv(chunk_path, index_col=0)
        else:
            print(f"  chunk {i + 1}/{len(chunks)} ({chunk_start} to {chunk_end}): fetching from Wind...")
            chunk_data = get_fund_nav_data(fund_codes, chunk_start, chunk_end, save_path=chunk_path, force_refresh=force_refresh)
            if chunk_data is None:
                print(
                    f"  Stopped at chunk {i + 1}/{len(chunks)} (see Wind error above, likely quota). "
                    f"{i} of {len(chunks)} chunks already completed and cached - rerun this cell "
                    f"later (e.g. after quota resets) to resume from here."
                )
                return None

        chunk_frames.append(chunk_data)

    fund_names = chunk_frames[0]["fund_name"]
    date_frames = [cf.drop(columns=["fund_name"]) for cf in chunk_frames]
    combined_dates = pd.concat(date_frames, axis=1)
    date_cols_sorted = sorted(combined_dates.columns)
    combined = pd.concat([fund_names, combined_dates[date_cols_sorted]], axis=1)

    print(f"  All {len(chunks)} chunks complete - {combined.shape[1] - 1} total trading days.")
    return combined


def get_index_daily_returns_chunked(index_code, start_date, end_date, save_prefix,
                                     chunk_months=CHUNK_MONTHS, force_refresh=False):
    """
    Same idea as get_fund_nav_data_chunked, for the benchmark index series.

    Each chunk's index is normalized to a plain "%Y-%m-%d" string right after
    it's fetched (or loaded from a per-chunk cache). Wind's w.wsd doesn't
    always return the same index dtype for every call - very short ranges
    (e.g. a 1-day chunk at the tail of a window) have come back indexed by
    plain datetime.date objects while other chunks come back as Timestamps,
    and a cache-loaded chunk is read back as plain strings. Concatenating
    frames with mixed index dtypes and then sort_index()-ing them raises
    TypeError: '<' not supported between instances of 'str' and
    'datetime.date' - normalizing every chunk up front avoids that.
    """
    chunks = date_chunks(start_date, end_date, chunk_months)
    chunk_frames = []

    for i, (chunk_start, chunk_end) in enumerate(chunks):
        chunk_path = f"{save_prefix}_chunk{i:03d}_{chunk_start}_{chunk_end}.csv"

        if not force_refresh and os.path.exists(chunk_path):
            print(f"  chunk {i + 1}/{len(chunks)} ({chunk_start} to {chunk_end}): loading from cache")
            chunk_data = pd.read_csv(chunk_path, index_col=0)
        else:
            print(f"  chunk {i + 1}/{len(chunks)} ({chunk_start} to {chunk_end}): fetching from Wind...")
            chunk_data = get_index_daily_returns(index_code, chunk_start, chunk_end, save_path=chunk_path, force_refresh=force_refresh)
            if chunk_data is None:
                print(
                    f"  Stopped at chunk {i + 1}/{len(chunks)} (see Wind error above, likely quota). "
                    f"{i} of {len(chunks)} chunks already completed and cached - rerun this cell "
                    f"later (e.g. after quota resets) to resume from here."
                )
                return None

        chunk_data = chunk_data.copy()
        chunk_data.index = [pd.Timestamp(idx).strftime("%Y-%m-%d") for idx in chunk_data.index]
        chunk_frames.append(chunk_data)

    combined = pd.concat(chunk_frames, axis=0).sort_index()
    combined = combined[~combined.index.duplicated(keep="last")]

    print(f"  All {len(chunks)} chunks complete.")
    return combined


def get_fund_nav_data_extended(fund_codes, gap_start, gap_end, recent_cache_path,
                                combined_save_path, chunk_months=CHUNK_MONTHS, force_refresh=False):
    """
    Extends an already-cached shorter NAV window (the existing 1-year cache)
    into a longer one by fetching ONLY the missing gap period from Wind, in
    small time chunks (see get_fund_nav_data_chunked), and concatenating it
    locally with the existing cache.
    """
    if combined_save_path and not force_refresh and os.path.exists(combined_save_path):
        print(f"Loading cached combined NAV data from {combined_save_path} (skip Wind fetch to save API quota)")
        return pd.read_csv(combined_save_path, index_col=0)

    if not os.path.exists(recent_cache_path):
        print(f"{recent_cache_path} not found - can't extend a cache that doesn't exist. Run the 1-year pipeline first.")
        return None
    recent_data = pd.read_csv(recent_cache_path, index_col=0)

    gap_prefix = (combined_save_path or "fund_nav_data_gap").replace(".csv", "")
    gap_data = get_fund_nav_data_chunked(
        fund_codes, gap_start, gap_end, gap_prefix, chunk_months=chunk_months, force_refresh=force_refresh
    )
    if gap_data is None:
        return None

    fund_names = recent_data["fund_name"]
    gap_date_cols = [c for c in gap_data.columns if c != "fund_name"]
    recent_date_cols = [c for c in recent_data.columns if c != "fund_name"]

    combined = pd.concat([gap_data[gap_date_cols], recent_data[recent_date_cols]], axis=1)
    date_cols_sorted = sorted(combined.columns)
    combined = pd.concat([fund_names, combined[date_cols_sorted]], axis=1)

    if combined_save_path:
        combined.to_csv(combined_save_path, index=True)

    return combined


def get_index_daily_returns_extended(index_code, gap_start, gap_end, recent_cache_path,
                                      combined_save_path, chunk_months=CHUNK_MONTHS, force_refresh=False):
    """Same idea as get_fund_nav_data_extended, for the benchmark index series."""
    if combined_save_path and not force_refresh and os.path.exists(combined_save_path):
        print(f"Loading cached combined index data from {combined_save_path} (skip Wind fetch to save API quota)")
        return pd.read_csv(combined_save_path, index_col=0)

    if not os.path.exists(recent_cache_path):
        print(f"{recent_cache_path} not found - can't extend a cache that doesn't exist. Run the 1-year pipeline first.")
        return None
    recent_data = pd.read_csv(recent_cache_path, index_col=0)
    recent_data.index = [pd.Timestamp(idx).strftime("%Y-%m-%d") for idx in recent_data.index]

    gap_prefix = (combined_save_path or "index_daily_returns_gap").replace(".csv", "")
    gap_data = get_index_daily_returns_chunked(
        index_code, gap_start, gap_end, gap_prefix, chunk_months=chunk_months, force_refresh=force_refresh
    )
    if gap_data is None:
        return None
    gap_data = gap_data.copy()
    gap_data.index = [pd.Timestamp(idx).strftime("%Y-%m-%d") for idx in gap_data.index]

    combined = pd.concat([gap_data, recent_data], axis=0).sort_index()
    combined = combined[~combined.index.duplicated(keep="last")]

    if combined_save_path:
        combined.to_csv(combined_save_path, index=True)

    return combined


### Merging data received from a colleague (optional, run once you have the files)

If someone else ran `fetch_gap_data_for_colleague.py` on a different Wind account and sent back the resulting CSVs, place them in this same folder and run the cell below. It merges each gap file with your existing 1-year cache into the exact `_3y.csv` files the pipeline cell below already looks for - so that cell will use them directly with zero Wind calls, regardless of your own quota.

In [ ]:
def merge_received_gap_nav(gap_csv_path, recent_cache_path, combined_save_path):
    """
    Combines a gap-period NAV CSV fetched by someone else (e.g. a colleague
    on a different Wind account, using fetch_gap_data_for_colleague.py) with
    the already-cached 1-year data - a pure local file merge, no Wind call.
    """
    if not os.path.exists(gap_csv_path):
        print(f"{gap_csv_path} not found - place the received file in this folder first.")
        return None
    if not os.path.exists(recent_cache_path):
        print(f"{recent_cache_path} not found - the 1-year pipeline needs to have run first.")
        return None

    gap_data = pd.read_csv(gap_csv_path, index_col=0)
    recent_data = pd.read_csv(recent_cache_path, index_col=0)

    fund_names = recent_data["fund_name"]
    gap_date_cols = [c for c in gap_data.columns if c != "fund_name"]
    recent_date_cols = [c for c in recent_data.columns if c != "fund_name"]

    combined = pd.concat([gap_data[gap_date_cols], recent_data[recent_date_cols]], axis=1)
    combined = pd.concat([fund_names, combined[sorted(combined.columns)]], axis=1)

    combined.to_csv(combined_save_path, index=True)
    print(f"Merged {gap_csv_path} + {recent_cache_path} -> {combined_save_path}")
    return combined


def merge_received_gap_index(gap_csv_path, recent_cache_path, combined_save_path):
    """Same idea as merge_received_gap_nav, for the benchmark index series."""
    if not os.path.exists(gap_csv_path):
        print(f"{gap_csv_path} not found - place the received file in this folder first.")
        return None
    if not os.path.exists(recent_cache_path):
        print(f"{recent_cache_path} not found - the 1-year pipeline needs to have run first.")
        return None

    gap_data = pd.read_csv(gap_csv_path, index_col=0)
    gap_data.index = [pd.Timestamp(idx).strftime("%Y-%m-%d") for idx in gap_data.index]

    recent_data = pd.read_csv(recent_cache_path, index_col=0)
    recent_data.index = [pd.Timestamp(idx).strftime("%Y-%m-%d") for idx in recent_data.index]

    combined = pd.concat([gap_data, recent_data], axis=0).sort_index()
    combined = combined[~combined.index.duplicated(keep="last")]

    combined.to_csv(combined_save_path, index=True)
    print(f"Merged {gap_csv_path} + {recent_cache_path} -> {combined_save_path}")
    return combined


# Run this once you've received the files back and placed them in this folder.
# Expects: fund_nav_data_{group}_gap.csv and index_daily_returns_{group}_gap.csv
# (exactly what fetch_gap_data_for_colleague.py produces) for each group below.
#
# This writes the SAME "_3y.csv" filenames the pipeline cell below already
# checks for as a cache - once these exist, that cell will use them directly
# and make ZERO Wind calls, regardless of your own account's quota.
for group_name in FUND_GROUPS:
    merge_received_gap_nav(
        gap_csv_path=f"fund_nav_data_{group_name.lower()}_gap.csv",
        recent_cache_path=f"fund_nav_data_{group_name.lower()}.csv",
        combined_save_path=f"fund_nav_data_{group_name.lower()}_3y.csv",
    )
    merge_received_gap_index(
        gap_csv_path=f"index_daily_returns_{group_name.lower()}_gap.csv",
        recent_cache_path=f"index_daily_returns_{group_name.lower()}.csv",
        combined_save_path=f"index_daily_returns_{group_name.lower()}_3y.csv",
    )


### Alternative: converting a wide-format file from a broader database

If what comes back is one wide file (dates as rows, fund codes as columns, covering close to the full 3-year span already) rather than matching our exact per-group gap format, use this instead of the `merge_received_gap_*` functions above. It intersects the file's fund codes with our tracked `fund_codes_*` lists - a broader/differently-screened database export won't have identical coverage to our hand-curated list, so this keeps only funds that are in both, and reports which of our funds aren't eligible (e.g. CSI2000/CNI2000 funds are expected to show zero matches, since those indices only launched in late 2023 - no fund tracking them can have data back to 2023-07-01).

In [ ]:
def convert_colleague_wide_nav(raw_path, save_suffix="_3y"):
    """
    Converts a wide-format NAV file (dates as rows, fund codes as columns -
    the shape a broader database export often comes in) into this project's
    per-group format. Keeps ONLY funds that are BOTH in our tracked
    fund_codes lists AND present in the received file - the file may cover a
    different/broader fund universe (e.g. it might only include funds with a
    full 3-year history, or come from a less-filtered database query), so
    this is an intersection, not a blind merge. Funds in our list that aren't
    in the file are reported as not eligible for the 3-year comparison; funds
    in the file that aren't in our list are ignored (we have no 1-year data
    for them, so they can't be compared across both horizons anyway).

    Saves directly as fund_nav_data_{group}_3y.csv per group, since a file
    like this typically already spans close to the full 3-year window
    (check the date range printed below against 2023-07-01/2026-07-01).
    """
    raw = pd.read_csv(raw_path, index_col=0)
    raw.index = [pd.Timestamp(idx).strftime("%Y-%m-%d") for idx in raw.index]
    print(f"Received file covers {raw.index.min()} to {raw.index.max()}")

    transposed = raw.T
    transposed.index.name = "fund_code"

    for group_name, group in FUND_GROUPS.items():
        fund_codes = group["fund_codes"]
        present = [c for c in fund_codes if c in transposed.index]
        missing = [c for c in fund_codes if c not in transposed.index]

        print(f"\n{group_name}: {len(present)}/{len(fund_codes)} of our tracked funds have confirmed 3-year data")
        if missing:
            print(f"  not eligible for 3-year comparison (no full history in received file): {missing}")

        if not present:
            print("  no eligible funds - skipping this group for the 3-year comparison.")
            continue

        recent_cache_path = f"fund_nav_data_{group_name.lower()}.csv"
        if not os.path.exists(recent_cache_path):
            print(f"  {recent_cache_path} not found - can't look up fund names. Run the 1-year pipeline first.")
            continue
        recent_data = pd.read_csv(recent_cache_path, index_col=0)
        fund_names = recent_data["fund_name"].reindex(present)

        group_data = transposed.loc[present].copy()
        group_data = pd.concat([fund_names.rename("fund_name"), group_data], axis=1)

        save_path = f"fund_nav_data_{group_name.lower()}{save_suffix}.csv"
        group_data.to_csv(save_path, index=True)
        print(f"  saved {len(present)} funds to {save_path}")


# Point this at wherever you saved the received file.
convert_colleague_wide_nav("colleague_nav_wide.csv")


### Benchmark 3-year data received as an Excel file (no Wind call needed)

If a colleague already sent back benchmark data as a wide-format Excel file (one column per index, covering the full 3-year window), use this instead of the direct-fetch cell below - it converts and splits the file into the same `index_daily_returns_{group}_3y.csv` files with zero Wind calls. Requires `openpyxl` (usually already included with Anaconda's pandas install).

In [ ]:
def convert_colleague_benchmark_excel(xlsx_path, save_suffix="_3y"):
    """
    Converts a wide-format benchmark Excel file (one column per index) into
    this project's per-group format (index=date, single PCT_CHG column).

    Handles the file having MULTIPLE stacked header blocks, not just one -
    e.g. 932000.CSI (CSI2000) launched later than the other indices, so a
    colleague pulling data by date range often has to fetch it separately
    and paste it as its own block below the main table, giving the sheet a
    second "Date" header row further down with a different column layout.
    Each block is parsed independently using its own header row to know how
    many columns/which codes belong to it.

    Only saves files for index codes actually used by a tracked FUND_GROUPS
    entry; any tracked index code not found in ANY block is reported at the
    end (expected to be harmless if that group also has no 3-year-eligible
    funds - nothing to benchmark against anyway).
    """
    raw = pd.read_excel(xlsx_path, sheet_name=0, header=None)
    header_rows = raw[raw[0] == "Date"].index.tolist()
    if not header_rows:
        print("No 'Date' header row found in this file - can't parse it.")
        return

    code_to_groups = {}
    for group_name, group in FUND_GROUPS.items():
        code_to_groups.setdefault(group["index_code"], []).append(group_name)

    found_codes = set()
    for block_num, header_row_idx in enumerate(header_rows):
        block_end = header_rows[block_num + 1] if block_num + 1 < len(header_rows) else len(raw)
        codes_in_block = [c for c in raw.iloc[header_row_idx].tolist()[1:] if pd.notna(c)]
        if not codes_in_block:
            continue

        block = raw.iloc[header_row_idx + 1:block_end, :len(codes_in_block) + 1].copy()
        block.columns = ["date"] + codes_in_block
        block["date"] = pd.to_datetime(block["date"], errors="coerce").dt.strftime("%Y-%m-%d")
        block = block.dropna(subset=["date"]).set_index("date")

        print(f"Block {block_num + 1}: rows {header_row_idx + 1}-{block_end - 1}, "
              f"codes {codes_in_block}, {len(block)} date rows.")

        for index_code in codes_in_block:
            found_codes.add(index_code)
            if index_code not in code_to_groups:
                print(f"  {index_code}: not used by any tracked group - skipping.")
                continue
            series = block[[index_code]].rename(columns={index_code: "PCT_CHG"}).astype(float)
            for group_name in code_to_groups[index_code]:
                save_path = f"index_daily_returns_{group_name.lower()}{save_suffix}.csv"
                series.to_csv(save_path, index=True)
                print(f"  {group_name} ({index_code}): saved {series.shape[0]} rows to {save_path}")

    missing_codes = [g["index_code"] for g in FUND_GROUPS.values() if g["index_code"] not in found_codes]
    if missing_codes:
        print(f"\nNot present in this file: {missing_codes}")


# Point this at wherever you saved the received Excel file.
convert_colleague_benchmark_excel("benchmark_3yr.xlsx")


In [ ]:
THREE_YEAR_START = "2023-07-01"
THREE_YEAR_END = "2026-07-01"

# Only 5 series total (one per unique index_code, none shared across these 5
# groups) - much smaller than the fund NAV fetch, so worth trying on your own
# account even if it couldn't handle the funds. Each is chunked internally
# (get_index_daily_returns_chunked) and cached per-chunk, so a partial
# failure is safe to resume by just rerunning this cell.

for group_name, group in FUND_GROUPS.items():
    index_code = group["index_code"]
    final_path = f"index_daily_returns_{group_name.lower()}_3y.csv"

    if os.path.exists(final_path):
        print(f"{group_name}: {final_path} already exists - skipping.")
        continue

    print(f"=== {group_name} ({index_code}) ===")
    combined = get_index_daily_returns_chunked(
        index_code, THREE_YEAR_START, THREE_YEAR_END,
        save_prefix=f"index_daily_returns_{group_name.lower()}_3y",
    )
    if combined is not None:
        combined.to_csv(final_path, index=True)
        print(f"  saved to {final_path}\n")
    else:
        print(f"  failed (see Wind error above, likely quota) - rerun this cell later to resume.\n")


In [ ]:
THREE_YEAR_START = "2023-07-01"  # kept for labeling/eligibility checks below
THREE_YEAR_END = "2026-07-01"

scorecards_3y = {}
nav_data_by_group_3y = {}
daily_returns_by_group_3y = {}
benchmark_returns_by_group_3y = {}
benchmark_returns_by_index_code_3y = {}

for group_name, group in FUND_GROUPS.items():
    fund_codes = group["fund_codes"]
    index_code = group["index_code"]

    if not fund_codes:
        continue

    recent_nav_cache = f"fund_nav_data_{group_name.lower()}.csv"  # existing 1-year cache
    nav_data = get_fund_nav_data_extended(
        fund_codes, GAP_START, GAP_END, recent_nav_cache,
        combined_save_path=f"fund_nav_data_{group_name.lower()}_3y.csv",
    )
    if nav_data is None:
        print(f"Skipping {group_name} (3y): failed to build extended NAV data.")
        continue

    if index_code in benchmark_returns_by_index_code_3y:
        benchmark_returns = benchmark_returns_by_index_code_3y[index_code]
    else:
        recent_bench_cache = f"index_daily_returns_{group_name.lower()}.csv"
        benchmark_returns = get_index_daily_returns_extended(
            index_code, GAP_START, GAP_END, recent_bench_cache,
            combined_save_path=f"index_daily_returns_{group_name.lower()}_3y.csv",
        )
        if benchmark_returns is not None:
            benchmark_returns_by_index_code_3y[index_code] = benchmark_returns

    if benchmark_returns is None:
        print(f"Skipping {group_name} (3y): failed to build extended benchmark data.")
        continue

    daily_returns = calculate_fund_daily_returns(
        nav_data, save_path=f"fund_daily_returns_{group_name.lower()}_3y.csv",
    )

    scorecard = build_fund_scorecard(
        nav_data, daily_returns, benchmark_returns,
        save_path=f"fund_scorecard_{group_name.lower()}_3y.csv",
    )
    scorecard.insert(0, "benchmark_group", group_name)

    first_date_col = daily_returns.columns[1]
    scorecard["has_full_3y_history"] = daily_returns[first_date_col].notna().reindex(scorecard.index)

    scorecards_3y[group_name] = scorecard
    nav_data_by_group_3y[group_name] = nav_data
    daily_returns_by_group_3y[group_name] = daily_returns
    benchmark_returns_by_group_3y[group_name] = benchmark_returns

if not scorecards_3y:
    print(
        "No group succeeded - master_scorecard_3y was not created. "
        "Check the Wind errors printed above (a connectivity failure will skip "
        "every group), fix the connection, and rerun this cell."
    )
else:
    master_scorecard_3y = pd.concat(scorecards_3y.values(), axis=0)
    master_scorecard_3y.to_csv("fund_scorecard_all_3y.csv", index=True)

    eligible_3y = master_scorecard_3y[master_scorecard_3y["has_full_3y_history"]]
    print(f"{eligible_3y.shape[0]} of {master_scorecard_3y.shape[0]} funds have a complete 3-year "
          f"history (inception before {THREE_YEAR_START}) - only these are eligible for the 3-year comparison.")


### Year-by-year IR persistence (3-year check)

The 1-year persistence check splits the period into two halves (H1/H2) -
reasonable when each half is still ~6 months. Over 3 years that same H1/H2
split gives each half ~1.5 years, which is coarse enough to hide a bad
stretch. Instead, check IR separately in each of the three individual
fiscal years (2023-07 to 2024-06, 2024-07 to 2025-06, 2025-07 to 2026-07) -
a fund good in every single year is a much stronger consistency signal
than one that's merely positive across two long chunks.

In [ ]:
def split_period_ir_by_year(daily_returns, benchmark_daily_returns, trading_days_per_year=242):
    """
    Information ratio computed separately for each of the three individual
    fiscal years in the 3-year window, rather than two coarse ~1.5-year
    halves. ir_persistence is the worst of the three - a fund only counts as
    "persistent" if it held up in every single year.
    """
    fund_names = daily_returns["fund_name"]
    excess_pct = calculate_excess_returns(daily_returns, benchmark_daily_returns)
    excess = excess_pct.drop(columns=["fund_name"]) / 100

    year_bounds = [
        ("Y1", "2023-07-01", "2024-06-30"),
        ("Y2", "2024-07-01", "2025-06-30"),
        ("Y3", "2025-07-01", "2026-07-01"),
    ]

    result = pd.DataFrame({"fund_name": fund_names})
    ir_cols = []
    for label, start, end in year_bounds:
        cols_in_range = [c for c in excess.columns if start <= c <= end]
        if not cols_in_range:
            continue
        sub = excess[cols_in_range]
        ann_excess = sub.mean(axis=1) * trading_days_per_year * 100
        tracking_error = sub.std(axis=1) * np.sqrt(trading_days_per_year) * 100
        result[f"ir_{label}"] = ann_excess / tracking_error
        ir_cols.append(f"ir_{label}")

    result["ir_persistence"] = result[ir_cols].min(axis=1)
    return result


In [ ]:
persistence_frames_3y = []
for group_name in daily_returns_by_group_3y:
    p = split_period_ir_by_year(daily_returns_by_group_3y[group_name], benchmark_returns_by_group_3y[group_name])
    p.insert(0, "benchmark_group", group_name)
    persistence_frames_3y.append(p)

persistence_table_3y = pd.concat(persistence_frames_3y, axis=0)
persistence_table_3y.to_csv("fund_ir_persistence_3y.csv", index=True)

best_funds_by_group_3y = select_best_funds(eligible_3y, persistence_table_3y, MIN_IR_BAR, MAX_PICKS)

year_ir_cols = [c for c in persistence_table_3y.columns if c.startswith("ir_Y")]
display_cols_3y = ["fund_name", "information_ratio", "ann_excess_return_pct",
                    "excess_max_drawdown_pct", "drawdown_ratio"] + year_ir_cols

for group_name, picks in best_funds_by_group_3y.items():
    display_name = GROUP_DISPLAY_NAMES.get(group_name, group_name)
    print(f"=== {display_name} (3-year): {len(picks)} fund(s) selected (IR > {MIN_IR_BAR}, positive every fiscal year, drawdown <= group median) ===")
    if len(picks):
        print(picks[display_cols_3y])
    else:
        print("No fund in this group clears the bar over 3 years.")
    print()


In [ ]:
comparison_rows = []
for group_name in FUND_GROUPS:
    one_yr_picks = set(best_funds_by_group.get(group_name, pd.DataFrame()).index)
    three_yr_picks = set(best_funds_by_group_3y.get(group_name, pd.DataFrame()).index)

    for fund_code in one_yr_picks | three_yr_picks:
        row = {"benchmark_group": group_name, "fund_code": fund_code}
        if fund_code in master_scorecard.index:
            row["fund_name"] = master_scorecard.loc[fund_code, "fund_name"]
            row["ir_1y"] = master_scorecard.loc[fund_code, "information_ratio"]
            row["excess_return_1y_pct"] = master_scorecard.loc[fund_code, "ann_excess_return_pct"]
            row["excess_dd_1y_pct"] = master_scorecard.loc[fund_code, "excess_max_drawdown_pct"]
        if fund_code in master_scorecard_3y.index:
            row["ir_3y"] = master_scorecard_3y.loc[fund_code, "information_ratio"]
            row["excess_return_3y_pct"] = master_scorecard_3y.loc[fund_code, "ann_excess_return_pct"]
            row["excess_dd_3y_pct"] = master_scorecard_3y.loc[fund_code, "excess_max_drawdown_pct"]
            # A 1-year-best pick without a full 3-year history will still show up
            # here with partial-period "3y" figures - flag it so it's not read
            # as a genuine 3-year track record.
            row["has_full_3y_history"] = master_scorecard_3y.loc[fund_code, "has_full_3y_history"]
        row["in_1y_best"] = fund_code in one_yr_picks
        row["in_3y_best"] = fund_code in three_yr_picks
        comparison_rows.append(row)

comparison_table = pd.DataFrame(comparison_rows).set_index("fund_code")
comparison_table = comparison_table.sort_values(["benchmark_group", "in_1y_best", "in_3y_best"], ascending=[True, False, False])

pd.set_option("display.max_rows", None)
print(comparison_table)
pd.reset_option("display.max_rows")

print()
overlap_count = ((comparison_table["in_1y_best"]) & (comparison_table["in_3y_best"])).sum()
print(f"{overlap_count} of {len(comparison_table)} candidate funds are best-in-class on BOTH the 1-year and 3-year view.")


## 3-year scorecard visualization

In [ ]:
group_order_3y = [g for g in FUND_GROUPS if g in daily_returns_by_group_3y]

fig, ax = plt.subplots(figsize=(10, 7))
colors_3y = plt.cm.tab10.colors

for i, group_name in enumerate(group_order_3y):
    subset = master_scorecard_3y[master_scorecard_3y["benchmark_group"] == group_name]
    ax.scatter(
        subset["tracking_error_pct"],
        subset["information_ratio"],
        s=subset["ann_vol_pct"] * 3,
        alpha=0.7,
        color=colors_3y[i % len(colors_3y)],
        label=GROUP_DISPLAY_NAMES.get(group_name, group_name),
    )

ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax.set_xlabel("Tracking Error (%)")
ax.set_ylabel("Information Ratio")
ax.set_title("3-Year: Information Ratio vs. Tracking Error (bubble size = annualized volatility)")
ax.legend(title="Benchmark Group")

plt.tight_layout()
plt.savefig("fund_ir_vs_te_scatter_3y.png", dpi=150)
plt.show()


In [ ]:
top15_3y = master_scorecard_3y.sort_values("information_ratio", ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top15_3y["fund_name"], top15_3y["information_ratio"], color="steelblue")
ax.invert_yaxis()
ax.set_xlabel("Information Ratio")
ax.set_title("Top Funds by 3-Year Information Ratio (across eligible groups)")
plt.tight_layout()
plt.savefig("fund_top15_ir_3y_bar.png", dpi=150)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(5 * len(group_order_3y) + 4, 6))

ir_by_group_3y = [
    master_scorecard_3y.loc[master_scorecard_3y["benchmark_group"] == g, "information_ratio"].dropna()
    for g in group_order_3y
]
excess_by_group_3y = [
    master_scorecard_3y.loc[master_scorecard_3y["benchmark_group"] == g, "ann_excess_return_pct"].dropna()
    for g in group_order_3y
]

axes[0].boxplot(ir_by_group_3y, labels=[GROUP_DISPLAY_NAMES.get(g, g) for g in group_order_3y])
axes[0].axhline(0, color="gray", linestyle="--", linewidth=0.8)
axes[0].set_title("3-Year Information Ratio Distribution by Group")
axes[0].set_ylabel("Information Ratio")

axes[1].boxplot(excess_by_group_3y, labels=[GROUP_DISPLAY_NAMES.get(g, g) for g in group_order_3y])
axes[1].axhline(0, color="gray", linestyle="--", linewidth=0.8)
axes[1].set_title("3-Year Annualized Excess Return Distribution by Group")
axes[1].set_ylabel("Annualized Excess Return (%)")

plt.tight_layout()
plt.savefig("group_comparison_3y_boxplots.png", dpi=150)
plt.show()


## 3-year drawdown analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

metrics_3y = [
    ("max_drawdown_pct", "NAV Max Drawdown (%)"),
    ("excess_max_drawdown_pct", "Excess-Return Max Drawdown (%)"),
    ("drawdown_ratio", "Drawdown Ratio (fund / benchmark)"),
]

for ax, (col, title) in zip(axes, metrics_3y):
    data_by_group = [
        master_scorecard_3y.loc[master_scorecard_3y["benchmark_group"] == g, col].dropna()
        for g in group_order_3y
    ]
    ax.boxplot(data_by_group, labels=[GROUP_DISPLAY_NAMES.get(g, g) for g in group_order_3y])
    if col == "drawdown_ratio":
        ax.axhline(1, color="gray", linewidth=0.8, linestyle="--")
    else:
        ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
    ax.set_title(f"3-Year {title}")

plt.tight_layout()
plt.savefig("drawdown_comparison_3y_by_group.png", dpi=150)
plt.show()


## 3-year best funds: table and charts

In [ ]:
best_funds_table_3y = pd.concat(
    [picks.assign(benchmark_group_display=GROUP_DISPLAY_NAMES.get(g, g))
     for g, picks in best_funds_by_group_3y.items() if len(picks)],
    axis=0,
) if any(len(p) for p in best_funds_by_group_3y.values()) else pd.DataFrame()

if best_funds_table_3y.empty:
    print("No fund in any group clears the 3-year selection bar.")
else:
    best_funds_table_3y = best_funds_table_3y.sort_values(
        ["benchmark_group_display", "information_ratio"], ascending=[True, False]
    )
    pd.set_option("display.max_rows", None)
    print(best_funds_table_3y[[
        "benchmark_group_display", "fund_name", "information_ratio",
        "ann_excess_return_pct", "excess_max_drawdown_pct", "drawdown_ratio",
    ]])
    pd.reset_option("display.max_rows")


In [ ]:
if not best_funds_table_3y.empty:
    labels_3y = [
        f"{row.fund_name} ({row.benchmark_group_display})"
        for row in best_funds_table_3y.itertuples()
    ]
    unique_groups_3y = list(best_funds_table_3y["benchmark_group_display"].unique())
    colors_map_3y = {g: c for g, c in zip(unique_groups_3y, plt.cm.tab10.colors)}
    bar_colors_3y = [colors_map_3y[g] for g in best_funds_table_3y["benchmark_group_display"]]

    fig, axes = plt.subplots(3, 1, figsize=(10, max(3, 0.4 * len(best_funds_table_3y)) * 3))

    panels = [
        ("information_ratio", "Information Ratio"),
        ("ann_excess_return_pct", "Annualized Excess Return (%)"),
        ("excess_max_drawdown_pct", "Excess-Return Max Drawdown (%)"),
    ]

    for ax, (col, title) in zip(axes, panels):
        ax.barh(labels_3y, best_funds_table_3y[col], color=bar_colors_3y)
        ax.invert_yaxis()
        ax.axvline(0, color="gray", linewidth=0.8)
        ax.set_title(f"Selected Best Funds (3-Year): {title}")

    legend_handles_3y = [mpatches.Patch(color=c, label=g) for g, c in colors_map_3y.items()]
    axes[0].legend(handles=legend_handles_3y, loc="lower right", fontsize=8)

    plt.tight_layout()
    plt.savefig("best_funds_selected_3y_charts.png", dpi=150)
    plt.show()


## 1-year vs. 3-year: who's only good in one period?

Splits the earlier `comparison_table` into three groups: best in 1-year only (may be a lucky recent stretch, not sustained), best in 3-year only (steady long-term performer that didn't stand out this particular year), and best in both (the most robust picks).

In [ ]:
both_periods = comparison_table.dropna(subset=["ir_1y", "ir_3y"])

fig, ax = plt.subplots(figsize=(8, 8))
colors_scatter = {g: c for g, c in zip(both_periods["benchmark_group"].unique(), plt.cm.tab10.colors)}

for group_name, subset in both_periods.groupby("benchmark_group"):
    ax.scatter(
        subset["ir_1y"], subset["ir_3y"],
        label=GROUP_DISPLAY_NAMES.get(group_name, group_name),
        color=colors_scatter[group_name], s=80, alpha=0.8,
    )

lims = [
    min(ax.get_xlim()[0], ax.get_ylim()[0]),
    max(ax.get_xlim()[1], ax.get_ylim()[1]),
]
ax.plot(lims, lims, color="gray", linestyle="--", linewidth=0.8, label="1yr IR = 3yr IR")
ax.axhline(0, color="gray", linewidth=0.5)
ax.axvline(0, color="gray", linewidth=0.5)
ax.set_xlabel("1-Year Information Ratio")
ax.set_ylabel("3-Year Information Ratio")
ax.set_title("1-Year vs 3-Year Information Ratio (candidate best funds)")
ax.legend()
plt.tight_layout()
plt.savefig("ir_1y_vs_3y_scatter.png", dpi=150)
plt.show()


In [ ]:
display_cols_compare = ["benchmark_group", "fund_name", "ir_1y", "ir_3y",
                        "excess_return_1y_pct", "excess_return_3y_pct"]

one_yr_only = comparison_table[comparison_table["in_1y_best"] & ~comparison_table["in_3y_best"]]
three_yr_only = comparison_table[comparison_table["in_3y_best"] & ~comparison_table["in_1y_best"]]
both_best = comparison_table[comparison_table["in_1y_best"] & comparison_table["in_3y_best"]]

print(f"=== Best in 1-YEAR ONLY, not sustained over 3 years - {len(one_yr_only)} fund(s) ===")
print(one_yr_only[display_cols_compare] if len(one_yr_only) else "(none)")

print(f"\n=== Best in 3-YEAR ONLY, didn't make this year's cut - {len(three_yr_only)} fund(s) ===")
print(three_yr_only[display_cols_compare] if len(three_yr_only) else "(none)")

print(f"\n=== Best in BOTH periods - most robust picks - {len(both_best)} fund(s) ===")
print(both_best[display_cols_compare] if len(both_best) else "(none)")


## Interactive fund-selection dashboard

Adjustable version of the fund-selection criteria used above. Every gate
is a control instead of a hardcoded constant:

- **Min IR** - information ratio floor (was fixed at 1.0 above)
- **Require persistence** - IR must be positive in every sub-period (both
  halves for 1-year, all three fiscal years for 3-year)
- **DD percentile** - keep only funds whose excess-return max drawdown is
  at or better than this percentile of their group (0 = off, 0.5 = median
  - what the fixed version above used, 0.75 = top quartile only)
- **TE percentile** - keep only funds whose tracking error is at or below
  this percentile of their group (1.0 = off - tracking error wasn't used
  as a gate at all above, 0.5 = median, 0.25 = tightest quartile only)
- **Min group size** - groups with fewer candidate funds than this skip
  *both* percentile gates (DD and TE) entirely and fall back to IR +
  persistence only. A percentile computed from 3 funds (e.g. CSI800) is
  noise, not signal - this stops a small group from being gated by a
  cutoff that isn't statistically meaningful
- **Max picks** - cap on picks per group

Move any slider and the tables + charts below redraw immediately.
Requires `ipywidgets` (`pip install ipywidgets`, then restart the kernel
if it's not already available).

In [ ]:
def select_best_funds_v2(scorecard, persistence, criteria):
    """
    Same idea as select_best_funds, but every gate is an adjustable knob in
    `criteria` instead of a hardcoded constant:
      min_ir               - IR must exceed this
      require_persistence  - require ir_persistence > 0
      dd_percentile        - keep funds with excess_max_drawdown_pct >= the
                              group's dd_percentile quantile (0 = off)
      te_percentile        - keep funds with tracking_error_pct <= the
                              group's te_percentile quantile (1.0 = off)
      min_group_size       - groups smaller than this skip BOTH percentile
                              gates entirely (IR + persistence only) - a
                              percentile from a handful of funds is noise
      max_picks            - cap on picks per group
    Pure computation (no printing/plotting), so it's cheap to call on every
    dashboard control change.
    """
    persistence_cols = [c for c in persistence.columns if c not in ("benchmark_group", "fund_name")]
    universe = scorecard.join(persistence[persistence_cols])

    selections = {}
    for group_name in scorecard["benchmark_group"].unique():
        group_universe = universe[universe["benchmark_group"] == group_name]
        group_size = len(group_universe)

        candidates = group_universe[group_universe["information_ratio"] > criteria["min_ir"]]

        if criteria["require_persistence"]:
            candidates = candidates[candidates["ir_persistence"] > 0]

        apply_percentile_gates = group_size >= criteria["min_group_size"]

        if apply_percentile_gates and criteria["dd_percentile"] > 0:
            dd_cutoff = group_universe["excess_max_drawdown_pct"].quantile(criteria["dd_percentile"])
            candidates = candidates[candidates["excess_max_drawdown_pct"] >= dd_cutoff]

        if apply_percentile_gates and criteria["te_percentile"] < 1.0:
            te_cutoff = group_universe["tracking_error_pct"].quantile(criteria["te_percentile"])
            candidates = candidates[candidates["tracking_error_pct"] <= te_cutoff]

        candidates = candidates.sort_values("information_ratio", ascending=False)
        selections[group_name] = candidates.head(criteria["max_picks"])

    return selections


In [ ]:
def render_dashboard_selection(scorecard, persistence, criteria, title_suffix=""):
    """Runs select_best_funds_v2, prints per-group tables, and draws two charts:
    selected funds ranked by IR, and IR vs. tracking error for the whole
    candidate pool with selected funds highlighted."""
    selections = select_best_funds_v2(scorecard, persistence, criteria)

    persistence_cols = [c for c in persistence.columns if c not in ("benchmark_group", "fund_name")]
    universe = scorecard.join(persistence[persistence_cols])

    total_picks = sum(len(p) for p in selections.values())
    ir_bar = criteria["min_ir"]
    print(f"Criteria: min_ir>{ir_bar:.2f}, require_persistence={criteria['require_persistence']}, "
          f"dd_percentile={criteria['dd_percentile']:.2f}, te_percentile={criteria['te_percentile']:.2f}, "
          f"min_group_size={criteria['min_group_size']}, max_picks={criteria['max_picks']}")
    print(f"{total_picks} fund(s) selected across {len(selections)} group(s) {title_suffix}\n")

    display_cols = ["fund_name", "information_ratio", "tracking_error_pct", "ann_excess_return_pct",
                     "excess_max_drawdown_pct", "drawdown_ratio"]

    for group_name in scorecard["benchmark_group"].unique():
        picks = selections.get(group_name, universe.iloc[0:0])
        display_name = GROUP_DISPLAY_NAMES.get(group_name, group_name)
        group_size = len(universe[universe["benchmark_group"] == group_name])
        gate_note = "" if group_size >= criteria["min_group_size"] else " (DD/TE percentile gates skipped - group too small)"
        print(f"=== {display_name}: {len(picks)} fund(s), {group_size} in candidate pool{gate_note} ===")
        if len(picks):
            print(picks[display_cols])
        else:
            print("No fund clears the current bar.")
        print()

    all_picks = pd.concat(
        [p.assign(benchmark_group_display=GROUP_DISPLAY_NAMES.get(g, g)) for g, p in selections.items() if len(p)],
        axis=0,
    ) if total_picks else pd.DataFrame()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    if not all_picks.empty:
        ordered = all_picks.sort_values("information_ratio", ascending=True)
        axes[0].barh(ordered["fund_name"], ordered["information_ratio"])
        axes[0].set_xlabel("Information Ratio")
        axes[0].set_title(f"Selected funds by IR {title_suffix}")
    else:
        axes[0].text(0.5, 0.5, "No funds selected", ha="center", va="center")
        axes[0].set_axis_off()

    axes[1].scatter(universe["tracking_error_pct"], universe["information_ratio"],
                     alpha=0.3, label="All candidates", color="gray")
    if not all_picks.empty:
        axes[1].scatter(all_picks["tracking_error_pct"], all_picks["information_ratio"],
                         color="crimson", label="Selected")
    axes[1].set_xlabel("Tracking Error (%)")
    axes[1].set_ylabel("Information Ratio")
    axes[1].set_title(f"IR vs. tracking error {title_suffix}")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    return selections


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

period_selector = widgets.Dropdown(options=[("1-year", "1y"), ("3-year", "3y")], value="1y", description="Period:")
min_ir_slider = widgets.FloatSlider(value=1.0, min=0.0, max=3.0, step=0.1, description="Min IR:")
persistence_checkbox = widgets.Checkbox(value=True, description="Require persistence")
dd_percentile_slider = widgets.FloatSlider(value=0.5, min=0.0, max=0.95, step=0.05, description="DD percentile:")
te_percentile_slider = widgets.FloatSlider(value=1.0, min=0.05, max=1.0, step=0.05, description="TE percentile:")
min_group_size_slider = widgets.IntSlider(value=8, min=1, max=15, description="Min group size:")
max_picks_slider = widgets.IntSlider(value=5, min=1, max=10, description="Max picks:")

_dashboard_controls = [
    period_selector, min_ir_slider, persistence_checkbox,
    dd_percentile_slider, te_percentile_slider, min_group_size_slider, max_picks_slider,
]
controls_box = widgets.VBox(_dashboard_controls)
dashboard_out = widgets.Output()


def _update_dashboard(*args):
    criteria = {
        "min_ir": min_ir_slider.value,
        "require_persistence": persistence_checkbox.value,
        "dd_percentile": dd_percentile_slider.value,
        "te_percentile": te_percentile_slider.value,
        "min_group_size": min_group_size_slider.value,
        "max_picks": max_picks_slider.value,
    }
    if period_selector.value == "1y":
        scorecard, persistence, suffix = master_scorecard, persistence_table, "(1-year)"
    else:
        scorecard, persistence, suffix = eligible_3y, persistence_table_3y, "(3-year)"

    with dashboard_out:
        clear_output(wait=True)
        render_dashboard_selection(scorecard, persistence, criteria, title_suffix=suffix)


for _control in _dashboard_controls:
    _control.observe(_update_dashboard, names="value")

display(controls_box, dashboard_out)
_update_dashboard()
